# Suite SVS — SystemVerilog simplified by IEEE 1800-2023

`SystemVerilog.Simplifiers.OfAny(term, scope)` simplifies a SystemVerilog term by IEEE 1800-2023's rules: 4-state
values through every operator, 11.6's sizing (context-determined and self-determined operands, sign extension) and
11.8's signedness. What the scope decides folds into the literal of its value, and the rest stays, the residual, each
subterm folded wherever its literal means what it did there: a self-determined subterm alone, a context-determined one
once its context's type is known, from the whole expression, the scope's values and the types it declares for the names
it does not bind. What has no rule stays too, named with its class in the evaluation inventory (`residues`): a write, a
class, a simulator's state (`d`), what waits on domain inference (`c`), what has no value of its own (`b`). Terms are
written through the dialect's writers and builders, in a transaction (`term`); `text` gives a residual's source, as the
printer writes it, and `shown` a value as its language writes it (a `Logic` as SystemVerilog writes a vector).

In [ ]:
import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import { Errors as FE, Terms as F } from "@mbse/expressions/Framework";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
import { Builders, Errors } from "@mbse/schemas/Framework";
import { assert, equal, raises } from "./support.js";

const { NotEvaluated } = Simplification;
const { ValueError } = Errors;
const { OverflowError } = FE;
const [SV, SVV, SVD, SVText, Formats] = [SystemVerilog.Expressions, SystemVerilog.Simplifiers, SystemVerilog.Domains,
  SystemVerilog.Text, SystemVerilog.Formats];

const store: any = new Builders.Store();
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, SystemVerilog.install, SVTypes.install,
  SVPatterns.install, SVAssertions.install, SVConstraints.install, SVPreprocessor.install]) {
  store.transaction(install);
}
store.transaction(SystemVerilog.define); // SystemVerilog's types, which its rules read through a term's holder

/** A port, a reference object: a class handle the scope may bind. */
function ports(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("svs.ports/1").mutations([(mu: any) => mu.upsert((w: any) => w.exists([(p: any) => p
    .symbols({ Port: S.OfObject })
    .spec([(_: any) => _.Port.name("Port").ref(true).properties({ width: S.OfInt })])]))]);
}

store.transaction(ports);
const port = store.transaction(() => store.Port((p: any) => p.width(8n)));

/** The SystemVerilog term `spec` writes, a native value as its literal, the root of a holder built in a transaction. */
function term(spec: unknown): any {
  return store.transaction(() => F.term(store.Expressions.Framework.Expression((e: any) => e.root(SV.DIALECT.spec(spec)))));
}

/** The simplification of the term `spec` writes, in `scope` (a `Scope`, or an object of variables). */
const simplified = (spec: unknown, scope: any = {}) => SVV.OfAny(term(spec), scope);

/** The source of the residual of the term `spec` writes, simplified in `scope`. */
const text = (spec: unknown, scope: any = {}): string => SVText.ToText(term(simplified(spec, scope).residual));

/** A value as its language writes it: a `Logic` as SystemVerilog writes a vector (`4'b10x1`). */
const shown = (found: unknown): unknown => (found instanceof SVD.Logic ? String(found) : found);

/** The value the term `spec` writes simplifies to, in `scope`, as `shown` writes it: its literal root's. */
const value = (spec: unknown, scope: any = {}): any => shown(simplified(spec, scope).evaluate());

/** The value SystemVerilog's folding rules give a term where no literal spells it (an array's): the term stays. */
function rule(spec: unknown, scope: any = {}): any {
  const found = SVV.RULES.evaluate(term(spec), scope instanceof SVV.Scope ? scope : new SVV.Scope(scope));
  return Array.isArray(found) ? found.map(shown) : shown(found);
}

/** Why what stayed in the residual stayed: each residue's reason and class. */
const reasons = (spec: unknown, scope: any = {}) => simplified(spec, scope).residues.map((r) => [r.reason, r.category]);

/** The value of `n` as a vector of `width` bits, as `shown` writes it: `integer`'s by default. */
function integer(n: bigint, width = 32, signed = true): string {
  const modulus = 1n << BigInt(width);
  return `${width}'${signed ? "s" : ""}b${(((n % modulus) + modulus) % modulus).toString(2).padStart(width, "0")}`;
}

const [x, y, q, s, p] = ["x", "y", "q", "s", "p"].map((name) => SV.identifier(name));
const [b, u, v, call] = [SV.binary, SV.unary, SV.vector, SV.call];
const [INTEGER, INT, REAL, SHORTREAL] = [SVD.named(store, "integer"), SVD.named(store, "int"), SVD.named(store, "real"),
  SVD.named(store, "shortreal")];
const atom = (keyword: string, signing: string | null = null) => (t: any) => t.integeratomtype(
  (a: any) => (signing === null ? a.keyword(keyword) : a.keyword(keyword).signing(signing))); // `int`, `int unsigned`
const dollar = (w: any) => w.dollarexpression((d: any) => d); // `$`
const null_ = (w: any) => w.constant((r: any) => r.singleton("null"));
const fill = (bit: string) => (w: any) => w.constant((r: any) => r.fill(bit)); // '0, '1, 'x, 'z
const this_ = (w: any) => w.thisexpression((t: any) => t);
const parenthesized = (inner: unknown) => (w: any) => w.parenthesizedexpression((r: any) => r.expression(SV.DIALECT.spec(inner)));

/** A time literal, `10ns`. */
const time = (magnitude: number, unit: string) => (w: any) => w.literal((r: any) => r.value((m: any) => m.as_scalar(magnitude)).unit(unit));

/** A package's scoped name, `package::name`; `$unit::name` where the package is null, `p::q::name` a pair. */
function scoped(pkg: string | [string, string] | null, name: string): unknown {
  const scope = (i: any) => {
    if (pkg === null) return i.unitname((d: any) => d);
    if (Array.isArray(pkg)) {
      return i.scopedname((c: any) => c.scope((j: any) => j.identifier((d: any) => d.spelling(pkg[0]))).name(
        (j: any) => j.identifier((d: any) => d.spelling(pkg[1]))));
    }
    return i.identifier((d: any) => d.spelling(pkg));
  };
  return (w: any) => w.nameexpression((r: any) => r.name((n: any) => n.scopedname((c: any) => c.scope(scope).name(
    (i: any) => i.identifier((d: any) => d.spelling(name))))));
}

/** A vector type of one packed dimension, `logic [left:right]`, signed or not. */
const vectorType = (keyword: string, left: bigint, right: bigint, signing: string | null = null) => (t: any) => t.integervectortype((n: any) => {
  n.keyword(keyword).dimensions([(d: any) => d.rangedimension((r: any) => r.left(SV.literal(left)).right(SV.literal(right)))]);
  return signing === null ? n : n.signing(signing);
});

<a id="SVS-01"></a>
## SVS-01 · Literals, names, `this`, `null` and a package's names fold into the literal of their value; a time literal by the scope's time unit

In [ ]:
// The root folds into its type's literal: an `integer`'s an unsized decimal integer, any other integral value a sized
// vector of its bits and signedness, a real a real; a literal stays as written.
assert(text(b("+", 1n, b("*", x, 2n)), { x: 3n }) === "7" && text(b("-", 1n, 8n)) === "-7");
assert(text(b("+", v("0011"), v("0001"))) === "4'b0100" && text(b("*", 1.5, 2n)) === "3.0" && text(v("10x1")) === "4'b10x1");
assert(value(x, { x: true }) === "1'b1" && value(x, { x: 2n ** 40n }) === integer(2n ** 40n, 64)); // a variable's type is its value's
// `null` is the null class handle, which compares with handles by identity; `this` is the scope's, by that name.
assert(simplified(null_).literal && simplified(null_).evaluate() === null && text(null_) === "null");
assert(text(b("==", x, y), { x: null }) === "null == y"); // a name's value, `null`, is its literal
const handle = new Map([["count", 4n]]); // a mapping: a structure, or a class handle
assert(value(b("==", x, null_), { x: null }) === "1'b1" && value(b("!=", x, null_), { x: handle }) === "1'b1");
assert(value(b("===", x, y), { x: handle, y: handle }) === "1'b1" && value(b("==", x, y), { x: handle, y: new Map([["count", 4n]]) }) === "1'b0");
assert(value(SV.member(this_, "count"), { this: handle }) === integer(4n) && text(SV.member(this_, "count")) === "this.count");
assert(value(b("==", x, x), { x: port }) === "1'b1" && value(SV.member(this_, "width"), { this: port }) === integer(8n)); // data
for (const spec of [b("==", x, 1n), b("==", 1n, x)]) raises(TypeError, () => value(spec, { x: handle }), "== expects a number, got dict"); // a handle is no number
// A package's names are its scope's (`packages`), `$unit::x` among them; one the scope does not provide stays, named.
const pkg = new SVV.Scope({ width: 8n }, { functions: { twice: (n: any) => 2n * n.integer() } });
const calls = (w: any) => w.callexpression((r: any) => r.callee(scoped("pkg", "twice")).arguments([SV.literal(21n)]));
assert(value(b("+", scoped("pkg", "width"), 1n), new SVV.Scope({}, { packages: { pkg } })) === integer(9n));
assert(value(calls, new SVV.Scope({}, { packages: { pkg } })) === integer(42n));
assert(value(scoped(null, "width"), new SVV.Scope({}, { packages: { $unit: pkg } })) === integer(8n) && text(scoped(null, "width")) === "$unit::width");
assert(equal(reasons(scoped("pkg", "width")), [["SystemVerilog scopedname is not simplified", "d"]])); // a class's, or unknown
assert(equal(reasons(scoped(["p", "q"], "x"), new SVV.Scope({}, { packages: { p: pkg } })), [["SystemVerilog scopedname is not simplified", "d"]]));
assert(text(b("+", y, scoped("pkg", "missing")), new SVV.Scope({}, { packages: { pkg } })) === "y + pkg::missing");
// A time literal is a `realtime` in the scope's time unit, rounded to its precision (3.14.2): 1, 10 or 100 of a unit.
assert(value(time(1.234, "ns"), new SVV.Scope({}, { timeunit: "1ns", timeprecision: "1ps" })) === 1.234);
assert(value(time(1.23456, "ns"), new SVV.Scope({}, { timeunit: "1ns", timeprecision: "10ps" })) === 1.23);
assert(value(time(1.5, "us"), new SVV.Scope({}, { timeunit: "100ps" })) === 15000.0 && value(time(3.0, "fs"), new SVV.Scope({}, { timeunit: "1ns" })) === 0.0);
assert(value(b("+", time(2.0, "ns"), 1n), new SVV.Scope({}, { timeunit: "1ns" })) === 3.0);
assert(equal(reasons(time(10.0, "ns")), [["SystemVerilog time literal without a time unit is not simplified", "a"]]));
for (const [options, message] of [[{ timeunit: "2ns" }, "a time unit is 1, 10 or 100 of s, ms, us, ns, ps or fs, got '2ns'"],
  [{ timeunit: "1ns", timeprecision: "10ns" }, "a time precision is at most its unit, got 10ns for 1ns"],
  [{ timeprecision: "1ps" }, "a time precision needs a time unit"],
  [{ mintypmax: "all" }, "min:typ:max's choice is min, typ or max, got 'all'"]] as [Record<string, string>, string][]) {
  raises(ValueError, () => new SVV.Scope({}, options), message);
}

<a id="SVS-02"></a>
## SVS-02 · Subterms fold where their meaning is kept: a self-determined one alone, a context-determined one once its context's type is known

In [ ]:
// A self-determined subterm folds alone: a concatenation's operand, a shift's count, an index, a condition, a cast's
// operand, a function's argument, a reduction's operand.
assert(text(SV.concatenation(y, b("+", v("0001"), v("0001")))) === "{y, 4'b0010}");
assert(text(b("<<", y, b("+", 1n, 2n))) === "y << 3" && text(SV.select(y, b("+", 1n, 1n))) === "y[2]");
assert(text(SV.conditional(b("<", 1n, 2n), y, x)) === "1'b1 ? y : x" && text(call("f", b("+", 1n, 2n), y)) === "f(3, y)");
assert(text(u("&", b("|", v("01"), v("10")))) === "1'b1" && text(b("&&", u("|", y), b("<", 1n, 2n))) === "|y && 1'b1");
// A context-determined one is computed in its context's type (11.6), so it folds once that type is known: here `y`'s,
// which the scope declares (`types`); unknown, `3 * 2` stays, since `y` could widen it.
assert(text(b("+", y, b("*", 3n, 2n))) === "y + 3 * 2");
assert(text(b("+", y, b("*", 3n, 2n)), new SVV.Scope({}, { types: { y: INTEGER } })) === "y + 6");
assert(text(b("+", y, b("+", 2147483647n, 1n)), new SVV.Scope({}, { types: { y: SVD.vector(64) } })) === "y + 64'b" + "0".repeat(32) + "1" + "0".repeat(31));
assert(text(b("+", y, b("-", 3n, 8n)), new SVV.Scope({}, { types: { y: SVD.vector(64) } })) === "y + 64'b" + "1".repeat(61) + "011"); // unsigned
assert(text(b("==", y, b("+", v("1111"), v("0001"))), new SVV.Scope({}, { types: { y: SVD.vector(5) } })) === "y == 5'b10000");
assert(text(b("+", y, b("+", 1n, 2n)), new SVV.Scope({}, { types: { y: SVD.vector(8) } })) === "y + 32'b" + "0".repeat(30) + "11");
// A type with no literal (2-state, `shortreal`) folds where its states cannot matter; where they could, it stays.
assert(text(b("+", SV.cast(atom("int"), b("+", 1n, 2n)), y), new SVV.Scope({}, { types: { y: INT } })) === "int'(3) + y");
assert(text(b("+", SV.cast(atom("int"), b("+", 1n, 2n)), y), new SVV.Scope({}, { types: { y: INTEGER } })) === "3 + y");
assert(text(b("&&", SV.cast(atom("int"), 3n), y)) === "32'sb" + "0".repeat(30) + "11 && y");
assert(text(b("+", x, y), { x: true }) === "x + y" && text(b("&&", x, y), { x: true }) === "1'b1 && y"); // a `bit`
assert(text(b("+", u("-", b("+", 1n, 2n)), y), new SVV.Scope({}, { types: { y: INTEGER } })) === "-3 + y");
assert(text(b("+", parenthesized(b("+", 1n, 2n)), y), new SVV.Scope({}, { types: { y: INTEGER } })) === "3 + y");
assert(text(call("f", parenthesized(x)), { x: handle }) === "f((x))"); // an object has no literal
assert(text(b("+", x, y), { x: 1.5 }) === "1.5 + y" && text(b("+", y, b("*", 1.5, 2n)), new SVV.Scope({}, { types: { y: REAL } })) === "y + 3.0");
const shortreal = (operand: unknown) => SV.cast((t: any) => t.nonintegertype((n: any) => n.keyword("shortreal")), operand);
assert(text(b("+", shortreal(0.1), y), new SVV.Scope({}, { types: { y: SHORTREAL } })) === "shortreal'(0.1) + y"); // no literal is one
assert(text(b("+", shortreal(0.1), y), new SVV.Scope({}, { types: { y: REAL } })) === "0.10000000149011612 + y");
// `**`'s exponent is self-determined where the power is integral, its context's where it is real.
assert(text(b("**", y, b("+", 1n, 1n))) === "y ** (1 + 1)" && text(b("**", y, b("+", 1n, 1n)), new SVV.Scope({}, { types: { y: INTEGER } })) === "y ** 2");
assert(text(b("**", y, b("+", 1n, 1n)), new SVV.Scope({}, { types: { y: REAL } })) === "y ** 2.0" && text(SV.replication(b("+", 1n, 1n), y)) === "{2{y}}");
// A name is its value's literal, of its own type, wherever it stands.
assert(text(b("+", x, y), { x: SVD.Logic.of("1x") }) === "2'b1x + y" && text(b("+", x, y), { x: 5n }) === "5 + y");
// `&&` and `||` are decided by an operand that decides them, `->` by a true consequent (Kleene's logic, x unknown).
assert(text(b("&&", y, 0n)) === "1'b0" && text(b("||", y, v("1"))) === "1'b1" && text(b("->", y, 1n)) === "1'b1");
assert(text(b("&&", y, 1n)) === "y && 1" && text(b("||", 0n, y)) === "0 || y");
// A conditional by a known condition, where the branch it keeps has the conditional's own type.
assert(text(SV.conditional(1n, y, b("+", 1n, 1n)), new SVV.Scope({}, { types: { y: INTEGER } })) === "y");
assert(text(SV.conditional(0n, v("1111"), y), new SVV.Scope({}, { types: { y: INTEGER } })) === "0 ? 4'b1111 : y");
assert(text(SV.conditional(1n, y, v("1111")), new SVV.Scope({}, { types: { y: INTEGER } })) === "1 ? y : 4'b1111");
assert(text(SV.conditional(0n, b("+", 1n, 1n), y), new SVV.Scope({}, { types: { y: INTEGER } })) === "y" && text(SV.conditional(1n, y, 2n)) === "1 ? y : 2");
assert(text(SV.conditional(v("x"), y, 1n), new SVV.Scope({}, { types: { y: INTEGER } })) === "1'bx ? y : 1");
// What a scope does not provide stays, as it was: a name, a function, and what they size.
const unbound = simplified(b("+", x, call("f", 1n)));
assert(text(b("+", x, call("f", 1n))) === "x + f(1)" && equal([...unbound.free].sort(), ["f", "x"]) && unbound.residues.length === 0);
raises(NotEvaluated, () => unbound.evaluate(), "the root is not a literal: f is not bound");

<a id="SVS-03"></a>
## SVS-03 · Casts: an integral type's operand sized as an assignment, vector types of constant dimensions, signings, `realtime` and `string`

In [ ]:
// A cast to an integral type takes its operand as an assignment would (6.24.1): computed in the wider width, by the
// operand's own signedness, then converted.
assert(value(SV.cast(16n, b("+", v("11111111"), v("00000001")))) === "16'b0000000100000000");
assert(value(SV.cast(atom("int"), b("+", v("11111111"), v("00000001")))) === integer(256n));
assert(value(SV.cast(4n, b("+", v("11111111"), v("00000001")))) === "4'b0000");
assert(value(SV.cast((t: any) => t.implicittype((i: any) => i.signing("signed")), b("+", v("1111"), v("0001")))) === "4'sb0000");
// Vector types of constant dimensions, signed or not, a keyword's type with a signing, `realtime` (`real`), `string`.
assert(value(SV.cast(vectorType("logic", 7n, 0n, "signed"), v("1111"))) === "8'sb00001111");
assert(value(SV.cast(vectorType("bit", 0n, 3n), v("1x01"))) === "4'b1001"); // 2-state: x is 0
assert(value(SV.cast((t: any) => t.integervectortype((n: any) => n.keyword("logic").dimensions([
  (d: any) => d.rangedimension((r: any) => r.left(b("-", 4n, 1n)).right(SV.literal(0n))),
  (d: any) => d.rangedimension((r: any) => r.left(SV.literal(1n)).right(SV.literal(0n)))])), -1n)) === "8'b11111111");
assert(value(SV.cast(atom("int", "unsigned"), -1n)) === integer(-1n, 32, false) && value(SV.cast(atom("byte", "unsigned"), 300n)) === "8'b00101100");
assert(value(SV.cast((t: any) => t.nonintegertype((n: any) => n.keyword("realtime")), 2n)) === 2.0);
const string_ = (t: any) => t.keywordtype((k: any) => k.keyword("string"));
assert(value(SV.cast(string_, v("0000000001000001"))) === "8'b01000001"); // its bytes, NUL bytes dropped
raises(TypeError, () => value(SV.cast(string_, 1.5)), "string' expects an integral operand, got float");
// The type's terms fold where they stand: a dimension's bound is self-determined.
assert(text(SV.cast((t: any) => t.integervectortype((n: any) => n.keyword("logic").dimensions([
  (d: any) => d.rangedimension((r: any) => r.left(b("-", 8n, 1n)).right(SV.literal(0n)))])), y)) === "logic [7:0]'(y)");
// A cast to a type with no value read here stays, named: a structure's, an enumeration's (domain inference, c), a
// typedef's name (its declaration, d), an unpacked dimension (b).
const named_ = (t: any) => t.namedtype((n: any) => n.name((i: any) => i.identifier((d: any) => d.spelling("word"))));
const unpacked = (t: any) => t.integervectortype((n: any) => n.keyword("logic").dimensions([
  (d: any) => d.sizedimension((r: any) => r.size(SV.literal(8n)))]));
assert(equal(reasons(SV.cast((t: any) => t.structtype((r: any) => r.keyword("struct")), 1n)), [
  ["SystemVerilog castexpression to structtype is not simplified", "c"]]));
assert(equal(reasons(SV.cast(named_, 1n)), [["SystemVerilog castexpression to namedtype is not simplified", "d"]]));
assert(equal(reasons(SV.cast(unpacked, 1n)), [["SystemVerilog Types sizedimension is not simplified", "b"]]));

<a id="SVS-04"></a>
## SVS-04 · Selects: a queue's last index `$`, slices of arrays and queues, and indexed part-selects of vectors and arrays

In [ ]:
// `$` is the last index of the array or the queue its select indexes (7.10.1).
assert(value(SV.select(q, dollar), { q: [1n, 2n, 3n] }) === integer(3n) && value(SV.select(q, b("-", dollar, 1n)), { q: [1n, 2n, 3n] }) === integer(2n));
assert(equal(rule(SV.range_(q, 1n, dollar), { q: [1n, 2n, 3n] }), [integer(2n), integer(3n)]));
assert(equal(rule(SV.range_(q, 2n, 1n), { q: [1n, 2n, 3n] }), []) && equal(rule(SV.range_(q, -1n, 7n), { q: [1n, 2n] }), [integer(1n), integer(2n)]));
assert(text(SV.select(q, dollar)) === "q[$]" && text(SV.select(y, b("-", dollar, b("+", 1n, 1n)))) === "y[$ - 2]");
raises(TypeError, () => value(SV.select(v("1010"), dollar)), "$ is the last index of a queue or an array, not of a vector");
// An indexed part-select (11.5.1): `[base +: width]`, bits from base up, `[base -: width]`, bits from base down, its
// width a constant; out of range x, and at an x base all x. Of an array or a queue, its items.
const part = (target: unknown, base: unknown, operator: string, width: bigint) => (w: any) => w.rangeselect(
  (r: any) => r.value(SV.DIALECT.spec(target)).left(SV.DIALECT.spec(base)).operator(operator).right(SV.literal(width)));
assert(value(part(v("11001010"), SV.literal(2n), "+:", 4n)) === "4'b0010" && value(part(v("11001010"), SV.literal(5n), "-:", 3n)) === "3'b001");
assert(value(part(v("1010"), SV.literal(2n), "+:", 4n)) === "4'bxx10" && value(part(v("1010"), v("x"), "+:", 2n)) === "2'bxx");
assert(equal(rule(part(q, SV.literal(1n), "+:", 2n), { q: [5n, 6n, 7n] }), [integer(6n), integer(7n)]));
assert(equal(rule(part(q, dollar, "-:", 2n), { q: [5n, 6n, 7n] }), [integer(6n), integer(7n)]));
assert(text(part(y, b("+", 1n, 1n), "+:", 4n)) === "y[2 +: 4]");
for (const [spec, error, message] of [
  [part(v("1010"), SV.literal(0n), "+:", 0n), ValueError, "a part-select's width must be positive, got 0"],
  [part(q, v("x"), "+:", 1n), ValueError, "a part-select's base must be known, got 1'bx"],
  [SV.range_(SV.literal(1.5), 1n, 0n), TypeError, "a float cannot be selected from"],
] as [unknown, Function, string][]) {
  raises(error, () => value(spec, { q: [1n] }), message);
}

<a id="SVS-05"></a>
## SVS-05 · `inside` with tolerance ranges `[a +/- b]` and `[a +%- b]`, and a range bounded by its type's `$`

In [ ]:
const tolerance = (a: unknown, operator: string, b_: unknown) => (w: any) => w.valuerange(
  (r: any) => r.left(SV.literal(a)).operator(operator).right(SV.literal(b_)));
assert(equal([7n, 8n, 12n, 13n].map((n) => value(SV.inside(n, tolerance(10n, "+/-", 2n)))), ["1'b0", "1'b1", "1'b1", "1'b0"]));
assert(equal([89n, 90n, 110n, 111n].map((n) => value(SV.inside(n, tolerance(100n, "+%-", 10n)))), ["1'b0", "1'b1", "1'b1", "1'b0"]));
assert(value(SV.inside(10.5, tolerance(10.0, "+%-", 5.0))) === "1'b1" && value(SV.inside(v("1x00"), tolerance(12n, "+/-", 1n))) === "1'bx");
// `$` bounds a range by its type's least or greatest value (11.4.13).
assert(value(SV.inside(v("11111111"), SV.span(v("00000010"), dollar))) === "1'b1");
assert(value(SV.inside(-5n, SV.span(dollar, 0n))) === "1'b1" && value(SV.inside(1.0e300, SV.span(0.0, dollar))) === "1'b1");
assert(value(SV.inside(-1.0e300, SV.span(dollar, 0.0))) === "1'b1");
assert(text(SV.inside(y, SV.span(b("+", 1n, 1n), dollar))) === "y inside {[1 + 1:$]}"); // its type is `y`'s too

<a id="SVS-06"></a>
## SVS-06 · `matches` and `&&&`: constant, wildcard and variable patterns, a variable bound in its conditional's consequence

In [ ]:
const matches = (value_: unknown, pattern: unknown) => (w: any) => w.matchesexpression(
  (m: any) => m.value(SV.DIALECT.spec(value_)).pattern(pattern));
const variable = (name: string) => (w: any) => w.variablepattern((p_: any) => p_.name((i: any) => i.spelling(name)));
const wildcard = (w: any) => w.wildcardpattern((c: any) => c);
const predicate = (...conditions: unknown[]) => (w: any) => w.predicateexpression(
  (r: any) => r.conditions(conditions.map((condition) => SV.DIALECT.spec(condition))));
const a = SV.identifier("a");
// A constant pattern matches a value identical to it (`===`); a wildcard and a variable any value (12.6).
assert(equal([5n, 6n].map((n) => value(matches(x, SV.literal(5n)), { x: n })), ["1'b1", "1'b0"]));
assert(value(matches(x, v("1x")), { x: SVD.Logic.of("1x") }) === "1'b1" && value(matches(x, wildcard), { x: 1n }) === "1'b1");
assert(value(matches(x, (w: any) => w.parenthesizedpattern((r: any) => r.pattern(SV.literal(5n)))), { x: 5n }) === "1'b1");
// A variable pattern binds the value in its conditional's consequence, and in the conditions after it in a `&&&`.
assert(value(SV.conditional(matches(x, variable("a")), b("+", a, 1n), 0n), { x: 41n }) === integer(42n));
const guarded = SV.conditional(predicate(matches(x, variable("a")), b(">", a, 2n)), a, 0n);
assert(value(guarded, { x: 9n }) === integer(9n) && value(guarded, { x: 1n }) === integer(0n));
assert(value(predicate(b(">", x, 0n), b("<", x, 9n)), { x: 3n }) === "1'b1" && value(predicate(v("x"), 1n)) === "1'bx");
assert(value(predicate(0n, v("x"))) === "1'b0");
assert(value(SV.conditional(predicate(matches(x, SV.literal(5n)), matches(x, variable("a"))), a, 0n), { x: 5n }) === integer(5n));
// What it binds stays bound: a `matches` with a variable stays as written, its consequence's names its own.
assert(text(SV.conditional(matches(y, variable("a")), b("+", a, b("+", 1n, 2n)), 0n)) === "y matches .a ? a + (1 + 2) : 0");
assert(text(SV.conditional(matches(y, SV.literal(5n)), b("+", 1n, 2n), 0n)) === "y matches 5 ? 3 : 0");
assert(text(matches(y, (w: any) => w.parenthesizedpattern((r: any) => r.pattern(b("+", 2n, 3n))))) === "y matches (2 + 3)");
assert(text(SV.conditional(predicate(b(">", y, 0n), matches(x, variable("a")), b(">", a, b("+", 1n, 1n))), a, 0n), { x: 5n })
  === "y > 0 &&& 5 matches .a &&& a > 2 ? a : 0");
assert(text(predicate(matches(y, variable("a")), b(">", a, b("+", 1n, 1n)))) === "y matches .a &&& a > 1 + 1");
// A tagged or a structure pattern waits on its tagged union's or its structure's type (c).
const tagged = (w: any) => w.taggedpattern((t: any) => t.member((i: any) => i.spelling("Valid")));
const structure = (w: any) => w.structurepattern((r: any) => r.items([wildcard]));
assert(equal(reasons(SV.conditional(matches(x, tagged), 1n, 0n), { x: 1n }), [
  ["SystemVerilog matchesexpression taggedpattern is not simplified", "c"]]));
assert(equal(reasons(matches(x, structure), { x: 1n }), [["SystemVerilog matchesexpression structurepattern is not simplified", "c"]]));

<a id="SVS-07"></a>
## SVS-07 · Streaming concatenations, `min:typ:max` by the scope's choice, and the empty queue

In [ ]:
/** `{<< slice {items}}`, each item an expression, or a pair of an array and its `with` range. */
function stream(operator: string, slice: unknown, ...items: unknown[]): unknown {
  const item = (it: unknown) => {
    if (Array.isArray(it)) {
      const [array, [left, operator_, right]] = it as [unknown, [bigint, string, bigint]];
      return (i: any) => i.expression(SV.DIALECT.spec(array)).left(SV.literal(left)).operator(operator_).right(SV.literal(right));
    }
    return (i: any) => i.expression(SV.DIALECT.spec(it));
  };
  return (w: any) => w.streamingconcatenation(
    (r: any) => (slice === null ? r.operator(operator) : r.operator(operator).slice(slice)).items(items.map(item)));
}

// `>>` streams the items' bits as they stand; `<<` takes slices from the right and places them left to right (11.4.14).
assert(value(stream("<<", SV.literal(4n), v("110101"))) === "6'b010111" && value(stream(">>", SV.literal(4n), v("110101"))) === "6'b110101");
assert(value(stream("<<", null, v("00110101"))) === "8'b10101100"); // a slice of 1 bit: the bits reversed
assert(value(stream("<<", atom("byte"), v("1010101011110000"))) === "16'b1111000010101010"); // a type's width
assert(value(stream("<<", SV.literal(2n), stream("<<", null, v("1101")))) === "4'b1110");
assert(value(stream(">>", null, q, v("1")), { q: [SVD.Logic.of("10"), SVD.Logic.of("0x")] }) === "5'b100x1"); // an array's items
const bits3 = { q: [SVD.Logic.of("1"), SVD.Logic.of("0"), SVD.Logic.of("1")] };
assert(value(stream(">>", null, [q, [1n, ":", 2n]]), bits3) === "2'b01" && value(stream(">>", null, [q, [0n, "+:", 2n]]), bits3) === "2'b10");
assert(value(stream(">>", null, [q, [2n, "-:", 2n]]), bits3) === "2'b01");
assert(text(stream("<<", b("+", 2n, 2n), y)) === "{<< 4 {y}}" && text(stream("<<", null, b("+", y, 1n))) === "{<<{y + 1}}");
assert(text(stream("<<", atom("byte"), y, b("+", v("01"), v("01")))) === "{<< byte {y, 2'b10}}");
for (const [spec, error, message] of [[stream("<<", SV.literal(0n), v("1")), ValueError, "a stream's slice must be positive, got 0"],
  [stream(">>", null, [v("1"), [0n, ":", 0n]]), TypeError, "a stream's `with` selects from an array, not a Logic"],
  [stream(">>", null, 1.5), TypeError, "a concatenation's parts must be integral, got float"]] as [unknown, Function, string][]) {
  raises(error, () => value(spec), message);
}
// `min:typ:max` is the expression the scope chooses (11.11), `typ` by default.
const choice = (w: any) => w.mintypmaxexpression((r: any) => r.min(SV.literal(1n)).typ(b("+", 1n, 1n)).max(SV.literal(3n)));
assert(equal(["min", "typ", "max"].map((chosen) => value(choice, new SVV.Scope({}, { mintypmax: chosen }))), [integer(1n), integer(2n), integer(3n)]));
assert(value(b("+", v("0001"), choice)) === integer(3n, 32, false)); // unsigned, as `4'b0001` is
assert(text((w: any) => w.mintypmaxexpression((r: any) => r.min(SV.literal(1n)).typ(y).max(b("+", 1n, 2n)))) === "1:y:1 + 2");
// The empty queue is an empty array, which no literal writes but `{}`.
const empty = (w: any) => w.emptyqueue((e: any) => e);
assert(equal(rule(empty), []) && text(empty) === "{}" && value(SV.method(empty, "size")) === integer(0n, 32));

<a id="SVS-08"></a>
## SVS-08 · Calls of the scope's functions: by position, by name and with defaults (`Subroutine`); a class handle's methods stay

In [ ]:
const three = new SVV.Subroutine((a_: any, b_: any, c_: any) => 100n * a_.integer() + 10n * b_.integer() + c_.integer(), ["a", "b", "c"],
  { b: 5n, c: 6n });
const scope = new SVV.Scope({ c: 7n }, { functions: { f: three, g: (n: any) => n } });

/** `.name(value)`, `.name()` or `.name`. */
const named = (name: string, value_: unknown = null, implicit = false) => (w: any) => w.namedconnection((c: any) => {
  c.name((i: any) => i.spelling(name));
  return value_ !== null ? c.value(SV.DIALECT.spec(value_)) : implicit ? c.implicit(true) : c;
});

const empty_ = (w: any) => w.emptyargument((e: any) => e);
const f = (fn: string, ...args: unknown[]) => (w: any) => w.callexpression(
  (r: any) => r.callee(SV.identifier(fn)).arguments(args.map((a_) => SV.DIALECT.spec(a_))));
assert(value(f("f", 1n, 2n, 3n), scope) === integer(123n) && value(f("f", 1n), scope) === integer(156n));
assert(value(f("f", 1n, empty_, 3n), scope) === integer(153n) && value(f("f", 1n, named("c", 9n)), scope) === integer(159n));
assert(value(f("f", named("a", 2n), named("b")), scope) === integer(256n) && value(f("f", 1n, named("c", null, true)), scope) === integer(157n));
assert(text(f("f", b("+", 1n, 1n), named("c", b("+", y, 1n))), scope) === "f(2, .c(y + 1))");
for (const [spec, message] of [[f("f"), "f's argument 'a' has no default"], [f("f", 1n, 2n, 3n, 4n), "f takes 3 arguments, its positional ones first"],
  [f("f", named("a", 1n), 2n), "f takes 3 arguments, its positional ones first"],
  [f("f", named("d", 1n)), "f has no formal 'd', or connects it twice"],
  [f("f", named("a", 1n), named("a", 2n)), "f has no formal 'a', or connects it twice"]] as [unknown, string][]) {
  raises(TypeError, () => value(spec, scope), message);
}
// A function the scope gives as a callable has no formals: named or empty arguments stay (a), as the call does.
assert(equal(reasons(f("g", named("n", 1n)), scope), [["SystemVerilog callexpression with a namedconnection is not simplified", "a"]]));
assert(equal(reasons(f("g", empty_), scope), [["SystemVerilog callexpression with an emptyargument is not simplified", "a"]]));
// Alone, a named connection or an empty argument has no value (b); a class handle's method needs its class (d).
assert(equal(reasons(named("a", 1n)), [["SystemVerilog namedconnection is not simplified", "b"]]));
assert(equal(reasons(SV.method(x, "start"), { x: handle }), [["SystemVerilog callexpression of a class handle's method is not simplified", "d"]]));
const local = (w: any) => w.nameexpression((r: any) => r.name((n: any) => n.localname((c: any) => c.name((i: any) => i.spelling("f")))));
assert(equal(reasons((w: any) => w.callexpression((r: any) => r.callee(local).arguments([]))), [
  ["SystemVerilog callexpression of a localname is not simplified", "d"]]));
assert(equal(reasons((w: any) => w.callexpression((r: any) => r.callee(parenthesized(SV.identifier("g"))).arguments([])), scope), [
  ["SystemVerilog callexpression of a parenthesizedexpression is not simplified", "a"]]));

<a id="SVS-09"></a>
## SVS-09 · Strings' methods (6.16): `len`, `getc`, `toupper`, `tolower`, `compare`, `icompare`, `substr` and the conversions

In [ ]:
const word = { s: "aBc" };
assert(value(SV.method(s, "len"), word) === integer(3n) && value(SV.method(SV.literal("hello"), "len")) === integer(5n));
assert(value(SV.method(s, "getc", 1n), word) === "8'sb01000010" && value(SV.method(s, "getc", 9n), word) === "8'sb00000000");
assert(value(SV.method(s, "toupper"), word) === value(SV.literal("ABC")) && value(SV.method(s, "tolower"), word) === value(SV.literal("abc")));
assert(value(SV.method(s, "substr", 1n, 2n), word) === value(SV.literal("Bc")) && value(SV.method(s, "substr", 2n, 1n), word) === "8'b00000000");
assert(value(SV.method(s, "compare", "abd"), word) === integer(-1n) && value(SV.method(s, "icompare", "ABC"), word) === integer(0n));
assert(value(SV.method(s, "compare", "aBc"), word) === integer(0n) && value(SV.method(s, "compare", "a"), word) === integer(1n));
for (const [method, text_, found] of [["atoi", "-1_234x", integer(-1234n)], ["atoi", "+12", integer(12n)], ["atoi", "x1", integer(0n)],
  ["atohex", "fF_g", integer(255n)], ["atooct", "778", integer(63n)], ["atobin", "1012", integer(5n)]]) {
  assert(value(SV.method(s, method as string), { s: text_ }) === found, `${method} ${text_}`);
}
assert(equal(["1_2.5e1z", "-.5", ".e3", ""].map((t) => value(SV.method(s, "atoreal"), { s: t })), [125.0, -0.5, 0.0, 0.0]));
assert(Formats.text(simplified(SV.method(s, "toupper"), { s: "é" }).evaluate() as any) === "Ã©"); // bytes, ASCII's letters alone change
for (const [spec, error, message] of [[SV.method(s, "getc"), TypeError, "getc() takes 1 arguments, got 0"],
  [SV.method(1.5, "len"), TypeError, "len() is a method of strings, not of float"],
  [SV.method(s, "compare", 1.5), TypeError, "compare() compares strings, not a float"],
  [SV.method(s, "getc", v("x")), ValueError, "getc's index must be a known integer, got 1'bx"]] as [unknown, Function, string][]) {
  raises(error, () => value(spec, word), message);
}
// The methods that write the string are a statement's (d).
assert(equal(reasons(SV.method(s, "putc", 0n, 65n), word), [["SystemVerilog callexpression method 'putc' is not simplified", "d"]]));

<a id="SVS-10"></a>
## SVS-10 · Arrays' locator methods (7.12.1), with or without `with`, and an item's `index()`; a method that writes stays

In [ ]:
const digits = { q: [3n, 1n, 4n, 1n, 5n] };
const odd = b("%", p, 2n);
const locate = (method: string, body: unknown) => SV.iterate(q, method, "p", body);
assert(equal(rule(locate("find", b(">", p, 1n)), digits), [integer(3n), integer(4n), integer(5n)]));
assert(equal(rule(locate("find_index", b(">", p, 1n)), digits), [integer(0n), integer(2n), integer(4n)]));
assert(equal(rule(locate("find_first", b(">", p, 3n)), digits), [integer(4n)]) && equal(rule(locate("find_first_index", b(">", p, 9n)), digits), []));
assert(equal(rule(locate("find_last", b("<", p, 4n)), digits), [integer(1n)]) && equal(rule(locate("find_last_index", b("<", p, 4n)), digits), [integer(3n)]));
assert(equal(rule(locate("min", odd), digits), [integer(4n)]) && equal(rule(locate("max", odd), digits), [integer(3n)]));
assert(equal(rule(locate("unique", odd), digits), [integer(3n), integer(4n)]) && equal(rule(locate("unique_index", odd), digits), [integer(0n), integer(2n)]));
assert(equal(rule(SV.method(q, "unique_index"), digits), [integer(0n), integer(1n), integer(2n), integer(4n)]));
assert(value(SV.method(locate("find", b(">", p, 1n)), "size"), digits) === integer(3n)); // a queue: its size folds
const itemIndex = (w: any) => w.callexpression((r: any) => r.callee(SV.member(p, "index")).arguments([]));
assert(equal(rule(locate("find", b("==", itemIndex, 1n)), { q: [7n, 8n, 9n] }), [integer(8n)]));
assert(value(locate("sum", b("*", p, itemIndex)), { q: [7n, 8n, 9n] }) === integer(26n));
assert(text(locate("find", b(">", p, b("+", 1n, 1n)))) === "q.find(p) with (p > 1 + 1)");
const notAMethod = (w: any) => w.arraymethodwithexpression((r: any) => r.value(call("f", p)).body(b("+", 1n, 2n)));
assert(text(notAMethod) === "f(p) with (3)" && equal(reasons(notAMethod), [
  ["SystemVerilog arraymethodwithexpression of a call that is not an array method's is not simplified", "a"]]));
// A method that writes the array is a statement's (d); any other the rules do not fold stays (a).
assert(equal(reasons(SV.method(q, "push_back", 1n), digits), [["SystemVerilog callexpression method 'push_back' is not simplified", "d"]]));
assert(equal(reasons(SV.method(q, "sum", 1n), digits), [["SystemVerilog callexpression method 'sum' is not simplified", "a"]]));
assert(equal(reasons(SV.method(x, "next"), { x: 1n }), [["SystemVerilog callexpression method 'next' is not simplified", "c"]]));

<a id="SVS-11"></a>
## SVS-11 · System functions: the math functions (C's library), the conversions between reals and bits, `$countbits`, `$isunbounded`

In [ ]:
// The math functions take reals, an integral argument converted, and give reals as C's library does: a domain error NaN,
// a pole or an overflow an infinity. Exact where IEEE 754 makes them so; the others are each host's library's.
assert(equal(["$sqrt", "$floor", "$ceil"].map((name) => value(call(name, 2n))), [Math.sqrt(2), 2.0, 2.0]));
assert(value(call("$pow", 2n, 10n)) === 1024.0 && value(call("$floor", -1.5)) === -2.0 && value(call("$ceil", 2.5)) === 3.0);
assert(Object.is(value(call("$ceil", -0.5)), -0) && value(call("$floor", Infinity)) === Infinity);
assert(equal(["$ln", "$log10"].map((name) => value(call(name, 0n))), [-Infinity, -Infinity]) && value(call("$atanh", 1n)) === Infinity);
assert(["$ln", "$log10", "$sqrt"].every((name) => Number.isNaN(value(call(name, -1n)))) && Number.isNaN(value(call("$asin", 2n))));
assert(Number.isNaN(value(call("$acosh", 0.5))) && Number.isNaN(value(call("$atanh", 2n))) && value(call("$atanh", -1n)) === -Infinity);
assert(value(call("$exp", 1000n)) === Infinity && value(call("$sinh", -1000n)) === -Infinity && value(call("$cosh", 1000n)) === Infinity);
assert(equal(["$exp", "$sin", "$cos", "$tan", "$asin", "$atan", "$sinh", "$cosh", "$tanh", "$asinh"].map((name) => value(call(name, 0n))), [
  1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0]));
assert(value(call("$acos", 1n)) === 0.0 && value(call("$acosh", 1n)) === 0.0 && value(call("$atan2", 0n, 1n)) === 0.0);
assert(value(call("$hypot", 3n, 4n)) === 5.0 && value(call("$log10", 1000n)) === 3.0 && value(call("$ln", 1n)) === 0.0);
raises(TypeError, () => value(call("$sqrt", x), { x: new Map() }), "$sqrt expects a number, got dict");
raises(TypeError, () => value(call("$pow", 2n)), "$pow takes 2 arguments, got 1");
// The conversions (20.5): `$rtoi` truncates to an `integer`, `$itor` converts one to a real; a real's encoding and back.
assert(value(call("$rtoi", -2.7)) === integer(-2n) && value(call("$rtoi", 3n)) === integer(3n) && value(call("$itor", v("11111111"))) === 255.0);
assert(value(call("$itor", 2.5)) === 3.0); // assigned to an integer first: rounded
assert(value(call("$realtobits", 1.0)) === "64'b0011111111110000" + "0".repeat(48) && value(call("$bitstoreal", call("$realtobits", -1.5))) === -1.5);
assert(value(call("$shortrealtobits", 1.0)) === "32'b00111111100000000000000000000000");
assert(value(call("$bitstoshortreal", call("$shortrealtobits", 0.1))) === 0.10000000149011612);
assert(value(b("+", call("$bitstoshortreal", v("00111111100000000000000000000000")), SV.cast((t: any) => t.nonintegertype(
  (n: any) => n.keyword("shortreal")), 0.1))) === 1.100000023841858); // a shortreal's sum, in binary32
raises(OverflowError, () => value(call("$rtoi", NaN)), "nan has no integer value");
raises(TypeError, () => value(call("$bitstoreal", 1.5)), "$bitstoreal expects an integral argument, got real");
// `$countbits` counts the bits in the states its controls name; `$isunbounded` tells `$`.
assert(value(call("$countbits", v("10xz10"), fill("1"), fill("x"))) === integer(3n) && value(call("$countbits", v("10xz10"), v("0"))) === integer(2n));
assert(value(call("$isunbounded", dollar)) === "1'b1" && value(call("$isunbounded", 1n)) === "1'b0");
raises(TypeError, () => value(call("$countbits", 1n)), "$countbits takes 2 arguments or more, got 1");
raises(TypeError, () => value(call("$countbits", 1n, 1.5)), "$countbits expects integral arguments, got float");

<a id="SVS-12"></a>
## SVS-12 · Array queries (20.7) of a value or a type: `$left`, `$right`, `$low`, `$high`, `$increment`, `$size`, `$dimensions`, `$unpacked_dimensions`

In [ ]:
const QUERIES = ["$left", "$right", "$low", "$high", "$increment", "$size", "$dimensions", "$unpacked_dimensions"];
const nested = { x: [[1n, 2n], [3n, 4n], [5n, 6n]] };
assert(equal(QUERIES.map((name) => value(call(name, x), nested)), [0n, 2n, 0n, 2n, -1n, 3n, 3n, 2n].map((n) => integer(n))));
assert(equal(QUERIES.slice(0, 6).map((name) => value(call(name, x, 2n), nested)), [0n, 1n, 0n, 1n, -1n, 2n].map((n) => integer(n))));
assert(equal(QUERIES.slice(0, 6).map((name) => value(call(name, x, 3n), nested)), [31n, 0n, 0n, 31n, 1n, 32n].map((n) => integer(n))));
assert(value(call("$left", x, 4n), nested) === integer(-1n).replace(/1/g, "x") && value(call("$size", x), { x: [] }) === integer(0n));
assert(equal(["$dimensions", "$left"].map((name) => value(call(name, x), { x: true })), [integer(0n), "32'sb" + "x".repeat(32)])); // a scalar
assert(equal(QUERIES.map((name) => value(call(name, vectorType("logic", 0n, 7n)))), [0n, 7n, 0n, 7n, -1n, 8n, 1n, 0n].map((n) => integer(n))));
assert(equal(["$left", "$size", "$dimensions"].map((name) => value(call(name, atom("byte")))), [integer(7n), integer(8n), integer(1n)]));
assert(value(call("$dimensions", (t: any) => t.nonintegertype((n: any) => n.keyword("real")))) === integer(0n));
assert(value(call("$dimensions", (t: any) => t.integervectortype((n: any) => n.keyword("logic")))) === integer(0n)); // a scalar
assert(value(call("$size", vectorType("logic", 7n, 0n))) === integer(8n));
raises(TypeError, () => value(call("$left", x, 1n, 1n), nested), "$left takes 1 or 2 arguments, got 3");
assert(equal(reasons(call("$left", (t: any) => t.integervectortype((n: any) => n.keyword("logic").dimensions([
  (d: any) => d.unsizeddimension((r: any) => r)])))), [["SystemVerilog Types unsizeddimension is not simplified", "b"]]));

<a id="SVS-13"></a>
## SVS-13 · `$sformatf` writes its arguments as its format says (21.2.1), reals as C writes them

In [ ]:
const written = (format: string, ...args: unknown[]) => Formats.text(simplified(call("$sformatf", format, ...args)).evaluate() as any);

assert(written("%d|%0d|%5d|%-5d|%05d", 5n, 5n, 5n, 5n, 5n) === "          5|5|    5|5    |00005");
assert(written("%d %d %0d", v("10x1"), v("xxxx"), SV.cast(8n, -1n)) === " X  x -1" && written("%d", v("zzzz")) === " z");
assert(written("%d", v("10z1")) === " Z" && written("%4h", v("0000")) === "0000");
assert(written("%h %0h %4h %b %o %x", v("00101x01"), v("00101101"), v("00101101"), v("101"), v("111000"), v("zzzz1z1z")) === "2X 2d 002d 101 70 zZ");
assert(written("%h %h", v("xxxx"), v("zzzz")) === "x z" && written("%s=%c%%", "ab", 65n) === "ab=A%" && written("%5s|%-3s|", "ab", "c") === "   ab|c  |");
assert(written("%e %f %g %.2f %10.3e %G", 1.5, 1.5, 0.0001, 2.675, 12345.678, 1e-10) === "1.500000e+00 1.500000 0.0001 2.67  1.235e+04 1e-10");
assert(written("%f %e %g %08.2f %-8.1f|", Infinity, -Infinity, NaN, -3.14159, 2.0) === "inf -inf nan -0003.14 2.0     |");
assert(written("%g %g %g %.3g", 100000.0, 1000000.0, 0.00001, 2.5) === "100000 1e+06 1e-05 2.5" && written("%f", 7n) === "7.000000");
assert(written("no specification") === "no specification" && written("%0d%%%0d", 1n, 2n) === "1%2");
// Formats.real writes a real as C does, from its exact value, rounded half to even: `%e`, `%.0e`, `%.3e`, then `%f`
// and `%g` alike.
for (const [number, written_] of [
  [0.0, "0.000000e+00 0e+00 0.000e+00 0.000000 0 0.000 0 0 0"],
  [-0.0, "-0.000000e+00 -0e+00 -0.000e+00 -0.000000 -0 -0.000 -0 -0 -0"],
  [0.5, "5.000000e-01 5e-01 5.000e-01 0.500000 0 0.500 0.5 0.5 0.5"],
  [2.5, "2.500000e+00 2e+00 2.500e+00 2.500000 2 2.500 2.5 2 2.5"],
  [0.1, "1.000000e-01 1e-01 1.000e-01 0.100000 0 0.100 0.1 0.1 0.1"],
  [1e-05, "1.000000e-05 1e-05 1.000e-05 0.000010 0 0.000 1e-05 1e-05 1e-05"],
  [123456.789, "1.234568e+05 1e+05 1.235e+05 123456.789000 123457 123456.789 123457 1e+05 1.23e+05"],
  [5e-324, "4.940656e-324 5e-324 4.941e-324 0.000000 0 0.000 4.94066e-324 5e-324 4.94e-324"],
  [9.9999995, "9.999999e+00 1e+01 1.000e+01 9.999999 10 10.000 10 1e+01 10"],
  [1e+16, "1.000000e+16 1e+16 1.000e+16 10000000000000000.000000 10000000000000000 10000000000000000.000 1e+16 1e+16 1e+16"],
] as [number, string][]) {
  assert(["e", "f", "g"].flatMap((c) => [null, 0, 3].map((p_) => Formats.real(c, number, p_))).join(" ") === written_, String(number));
}
for (const [spec, error, message] of [[call("$sformatf", "%d"), TypeError, "$sformatf's format has more specifications than arguments"],
  [call("$sformatf", "", 1n), TypeError, "$sformatf's format has fewer specifications than arguments"],
  [call("$sformatf", "%q", 1n), ValueError, "%q is not a format specification"],
  [call("$sformatf", "%d", 1.5), TypeError, "%d expects an integral argument, got real"],
  [call("$sformatf", 1.5), TypeError, "$sformatf's format is a string, not a float"],
  [call("$sformatf", "%d", x), TypeError, "$sformatf writes numbers and strings, not a dict"],
  [call("$sformatf"), TypeError, "$sformatf takes 1 argument or more, got 0"]] as [unknown, Function, string][]) {
  raises(error, () => simplified(spec, { x: new Map() }), message);
}
// What reads a simulator's settings or hierarchy stays (d); a value's encoding or its assignment pattern (a).
assert(equal(reasons(call("$sformatf", "%t", 1n)), [["SystemVerilog systemcall '$sformatf' %t is not simplified", "d"]]));
assert(equal(reasons(call("$sformatf", "%p", 1n)), [["SystemVerilog systemcall '$sformatf' %p is not simplified", "a"]]));

<a id="SVS-14"></a>
## SVS-14 · What stays, by its class: `FOLDED`, `CATEGORIES`, and each kind no rule folds named in the residual

In [ ]:
assert(SVV.FOLDED.size === 26 && ["dollarexpression", "streamingconcatenation", "mintypmaxexpression", "emptyqueue",
  "namedconnection", "emptyargument", "matchesexpression", "predicateexpression", "thisexpression"].every((kind) => SVV.FOLDED.has(kind)));
assert(equal(SV.TAGS.filter((tag) => !SVV.FOLDED.has(tag)).sort(), ["incrementexpression", "assignmentexpression", "valuerange",
  "assignmentpattern", "patternitem", "superexpression", "newexpression", "newcopyexpression", "newarrayexpression",
  "randomizewithexpression", "taggedexpression", "rootexpression"].sort()));
for (const [spec, found] of [
  [(w: any) => w.incrementexpression((r: any) => r.operator("++").operand(x)), ["SystemVerilog incrementexpression is not simplified", "d"]],
  [(w: any) => w.assignmentpattern((r: any) => r.items([SV.literal(1n)])), ["SystemVerilog assignmentpattern is not simplified", "c"]],
  [(w: any) => w.taggedexpression((r: any) => r.member((i: any) => i.spelling("Valid"))), ["SystemVerilog taggedexpression is not simplified", "c"]],
  [(w: any) => w.rootexpression((r: any) => r), ["SystemVerilog rootexpression is not simplified", "d"]],
  [SV.span(1n, 2n), ["SystemVerilog valuerange is not simplified", "b"]],
  [call("$time"), ["SystemVerilog systemcall '$time' is not simplified", "d"]],
  [call("$typename", 1n), ["SystemVerilog systemcall '$typename' is not simplified", "c"]],
  [(w: any) => w.nameexpression((r: any) => r.name((n: any) => n.localname((c: any) => c.name((i: any) => i.spelling("x"))))),
    ["SystemVerilog nameexpression of a localname is not simplified", "d"]],
  [call("$bits", (t: any) => t.keywordtype((k: any) => k.keyword("string"))), ["SystemVerilog systemcall '$bits' of keywordtype is not simplified", "b"]],
] as [unknown, string[]][]) {
  assert(equal(reasons(spec, { x: 1n }), [found]), found[0]);
}
// Within a sublanguage's term, which no rule reads, an operand's context is the sublanguage's: what passes it on stays.
const firstMatch = (inner: unknown) => (w: any) => w.firstmatchsequence((r: any) => r.sequence(SV.DIALECT.spec(inner)));
assert(text(call("f", firstMatch(b("+", y, b("+", 1n, 2n))), SV.concatenation(b("+", v("01"), v("01"))))) === "f(first_match(y + (1 + 2)), 2'b10)");
assert(text(call("f", firstMatch(SV.conditional(1n, y, 2n)))) === "f(first_match(1 ? y : 2))");
// A residue does not keep what does not depend on it from folding around it.
assert(text(b("+", (w: any) => w.incrementexpression((r: any) => r.operator("++").operand(x)), b("+", 1n, 2n))) === "++x + (1 + 2)");
assert(text(SV.concatenation((w: any) => w.incrementexpression((r: any) => r.operator("++").operand(x)), b("+", v("01"), v("01")))) === "{++x, 2'b10}");